## Nome: Caio Semblano Silva
## Matrícula: 621
-----


# ATIVIDADE 9 — Integração Numérica: Simpson (1/3) e Simpson Repetida

**Objetivo:** praticar a aplicação da Regra de Simpson e da Regra de Simpson Repetida (composta) para aproximar integrais definidas e áreas a partir de dados tabulados, aplicando em problemas baseados em projetos da FETIN (Feira Tecnológica do Inatel).

---


## Resumo Teórico

**Regra de Simpson (1/3)**: aproxima a integral por um **polinômio quadrático** que interpola a função em três pontos igualmente espaçados $x_0, x_1, x_2$ com passo $h = \tfrac{b-a}{2}$:

$$\int_a^b f(x)\,dx \approx \frac{h}{3}\,[f(x_0) + 4f(x_1) + f(x_2)].$$

**Regra de Simpson repetida (composta)**: divide-se $[a,b]$ em $n$ subintervalos **pares** (passo $h=\tfrac{b-a}{n}$) e aplica-se Simpson a cada par de subintervalos. Fórmula compacta:

$$\int_a^b f(x)\,dx \approx \frac{h}{3}\Big[ f(x_0) + 4\sum_{\text{índice ímpar}} f(x_i) + 2\sum_{\text{índice par interno}} f(x_i) + f(x_n) \Big],\quad n\ \text{par}.$$


## Funções-base

In [5]:

from typing import Callable, Sequence
import numpy as np

def simpson_simples(f: Callable[[float], float], a: float, b: float) -> float:
    """Regra de Simpson simples (usa apenas 3 pontos)."""
    h = (b - a) / 2.0
    x0, x1, x2 = a, (a + b) / 2.0, b
    return (h / 3.0) * (f(x0) + 4.0 * f(x1) + f(x2))

def simpson_repetida(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    """Regra de Simpson repetida (composta). Requer n PAR."""
    if n % 2 != 0:
        raise ValueError("n deve ser PAR para Simpson repetida.")
    h = (b - a) / n
    s = f(a) + f(b)
    # Índices ímpares
    s += 4.0 * sum(f(a + (2*k - 1)*h) for k in range(1, n//2 + 1))
    # Índices pares internos
    s += 2.0 * sum(f(a + 2*k*h)     for k in range(1, n//2))
    return s * h / 3.0

def simpson_repetida_tabulado(y: Sequence[float], h: float) -> float:
    """Simpson composta para DADOS TABULADOS igualmente espaçados.
    y: sequência [y0, y1, ..., yn] com n PAR (ou seja, número de subintervalos n = len(y)-1 é par).
    h: espaçamento entre abscissas.
    """
    n = len(y) - 1
    if n % 2 != 0:
        raise ValueError("len(y)-1 deve ser PAR para Simpson repetida.")
    y0_yn = y[0] + y[-1]
    soma_impares = sum(y[i] for i in range(1, n, 2))
    soma_pares   = sum(y[i] for i in range(2, n, 2))
    return (h/3.0) * (y0_yn + 4.0*soma_impares + 2.0*soma_pares)


**Questão 1 – Simpson (composta) em função analítica**

Para a FETIN deste ano, uma equipe desenvolveu um robô autônomo (Inatel Maker) para entregas no campus. Durante os testes de bancada, a equipe modelou a potência elétrica $P(t)$ (em Watts) consumida pelo motor de tração durante o primeiro segundo de aceleração como:

$$
P(t) = \frac{4}{1+t^2}
$$

Para dimensionar a bateria, eles precisam calcular a energia total consumida neste intervalo de tempo, dada pela integral:

$$
E = \int_{0}^{1} \frac{4}{1+t^2} \,dt
$$

Calcule esta integral usando a **Regra de Simpson** com **4 subintervalos**.  
Arredonde o resultado para **3 casas decimais**.

**Dica:**  
Use `simpson_repetida(f, a=0, b=1, n=4)`.  
Depois, compare com o valor analítico (a energia real consumida):

$$
4 \arctan(t)\Big\vert{}_{0}^{1}
$$

In [7]:
def P(t):
    return 4 / (1 + t**2)

energia_simples = simpson_simples(P, a=0, b=1)
energia_simpson = simpson_repetida(P, a=0, b=1, n=4)
energia_analitica = 4 * (np.arctan(1) - np.arctan(0))

print(f"{'Método':<28} | {'Energia (J)':<14} | Erro absoluto")
print("-" * 62)
print(f"{'Simpson simples (3 pontos)':<28} | {energia_simples:<14.6f} | {abs(energia_analitica - energia_simples):.6e}")
print(f"{'Simpson repetida (n=4)':<28} | {energia_simpson:<14.6f} | {abs(energia_analitica - energia_simpson):.6e}")
print(f"{'Analítico (4 arctan t)':<28} | {energia_analitica:<14.10f} | 0 (exato)")
print()
print(f"Resultado pedido (Simpson, n=4, 3 casas): {energia_simpson:.3f}")


Método                       | Energia (J)    | Erro absoluto
--------------------------------------------------------------
Simpson simples (3 pontos)   | 3.133333       | 8.259320e-03
Simpson repetida (n=4)       | 3.141569       | 2.402614e-05
Analítico (4 arctan t)       | 3.1415926536   | 0 (exato)

Resultado pedido (Simpson, n=4, 3 casas): 3.142


Qual o método mais exato?

**Resposta:**
O método mais exato é o **analítico**. A integral $\int_0^1 4/(1+t^2)\,dt$ vale exatamente $4\arctan(t)\big|_{0}^{1} = \pi$.

Entre as aproximações numéricas, a **Simpson repetida** ($n=4$) é bem mais precisa do que a Simpson simples: o erro cai de cerca de $8{,}3\times 10^{-3}$ para $2{,}4\times 10^{-5}$. Isso acontece porque a regra composta usa mais pontos e aproxima a curva por trechos quadráticos menores, **Resposta:**
O método mais exato é o **analítico**. A integral $\int_0^1 4/(1+t^2)\,dt$ vale exatamente $4\arctan(t)\big|_{0}^{1} = \pi$.

Entre as aproximações numéricas, a **Simpson repetida** ($n=4$) é bem mais precisa do que a Simpson simples: o erro cai de cerca de $8{,}3\times 10^{-3}$ para $2{,}4\times 10^{-5}$. Isso acontece porque a regra composta usa mais pontos e aproxima a curva por trechos quadráticos menores, em vez de uma única parábola no intervalo inteiro.

**Questão 2 – Simpson Repetida (composta)**

Uma equipe da área de Telecomunicações apresentou na FETIN um estudo sobre o impacto do ruído térmico em redes 5G. A densidade de probabilidade do ruído em um de seus circuitos de recepção foi modelada por uma distribuição normal padrão. A probabilidade de o ruído estar contido em uma faixa específica de operação é dada pela integral:

$$
P = \int_{0}^{3} \frac{1}{\sqrt{2\pi}} e^{-\frac{x^2}{2}} \,dx
$$

Calcule esta integral usando a **Regra de Simpson Repetida** com **100 subintervalos (ISR)**. <br>
Arredonde o resultado para 7 casas decimais.  
Não é necessário calcular o erro de truncamento.

Dica: Use `np.sqrt()`, `np.pi` e `np.exp()` quando necessário.


In [11]:
def f(x):
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-x**2 / 2)

P_ruido = simpson_repetida(f, a=0, b=3, n=100)

print(f"Probabilidade (Simpson repetida, n=100): {P_ruido:.7f}")

Probabilidade (Simpson repetida, n=100): 0.4986501



**Questão 3 – Área de seção transversal (Dados Tabulados)**
Um projeto de IoT voltado para o agronegócio, destaque na FETIN, propõe o uso de um drone subaquático para mapear canais de irrigação e otimizar o uso da água na lavoura. O drone mediu a profundidade d(x) de um canal em intervalos de 0,5 metro, cobrindo uma largura total de 2,0 metros.

Considere as abscissas x = [0.0, 0.5, 1.0, 1.5, 2.0] (em metros) e as profundidades d(x) (em metros) correspondentes registradas pelo sensor do drone:

d = [0.0, 1.7, 2.5, 0.9, 0.5]

A área da seção transversal do canal é necessária para calcular a vazão de água. Calcule a área usando:


1. Simpson (simples): use apenas x=\{0\;1\;2\}, 3 pontos igualmente espaçados, h=1.  

In [14]:
# Pontos x = {0, 1, 2} → profundidades d = {0.0, 2.5, 0.5}, com h = 1
h_simples = 1.0
d_simples = [0.0, 2.5, 0.5]
area_simples = simpson_repetida_tabulado(d_simples, h_simples)

print(f"Área (Simpson simples, h=1): {area_simples:.2f} m²")

Área (Simpson simples, h=1): 3.50 m²



2. Simpson Repetida (composta): use todos os 5 pontos com h=0,5 (número de subintervalos n=4, que é par).

In [16]:
# Todos os 5 pontos, h = 0,5 e n = 4 (par)
h_composta = 0.5
d = [0.0, 1.7, 2.5, 0.9, 0.5]
area_composta = simpson_repetida_tabulado(d, h_composta)

print(f"Área (Simpson repetida, h=0.5): {area_composta:.2f} m²")
print()
print("Comparação das áreas:")
print(f"  Simpson simples:  {area_simples:.2f} m²")
print(f"  Simpson repetida: {area_composta:.2f} m²")

Área (Simpson repetida, h=0.5): 2.65 m²

Comparação das áreas:
  Simpson simples:  3.50 m²
  Simpson repetida: 2.65 m²
